# Homework 05: Data storage
Name: Paritosh Dwivedi
Date: August 18, 2026

Objectives:
- Env-driven paths to `data/raw/` and `data/processed/`
- Save CSV and Parquet; reload and validate
- Abstract IO with utility functions; document choices

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas
# !pip install pyarrow
# !pip install python-dotenv

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/paritoshdwivedi/Downloads/project bootcamp/bootcamp_paritosh_dwivedi/homework/homework05

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [3]:
import datetime as dt
import importlib.util
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()
raw_setting = os.getenv("DATA_DIR_RAW")
processed_setting = os.getenv("DATA_DIR_PROCESSED")
if not raw_setting or not processed_setting:
    raise RuntimeError("DATA_DIR_RAW and DATA_DIR_PROCESSED must be set in .env.")

RAW = Path(raw_setting)
PROC = Path(processed_setting)
for configured_path in (RAW, PROC):
    if configured_path.is_absolute() or ".." in configured_path.parts:
        raise ValueError("Data directories must be relative to the homework folder.")
    configured_path.mkdir(parents=True, exist_ok=True)

print("RAW ->", RAW)
print("PROC ->", PROC)

RAW -> data/raw
PROC -> data/processed


## 1) Create a sample DataFrame
This small typed table mirrors the Weekly ETF Risk Monitor's weekly review output for SPY. A real datetime column, float measures, and categorical labels make the storage comparison meaningful.

In [4]:
import numpy as np
dates = pd.date_range("2026-08-10", periods=6, freq="B")
df = pd.DataFrame(
    {
        "date": dates,
        "ticker": pd.Categorical(["SPY"] * len(dates)),
        "next_five_session_vol_forecast": np.array(
            [0.128, 0.124, 0.121, 0.117, 0.114, 0.113], dtype="float64"
        ),
        "elevated_risk_score": np.array(
            [0.312, 0.298, 0.284, 0.271, 0.259, 0.252], dtype="float64"
        ),
        "risk_flag": pd.Categorical(
            ["normal"] * len(dates), categories=["normal", "elevated"]
        ),
    }
)

print(df.dtypes)
df

date                              datetime64[ns]
ticker                                  category
next_five_session_vol_forecast           float64
elevated_risk_score                      float64
risk_flag                               category
dtype: object


,date,ticker,next_five_session_vol_forecast,elevated_risk_score,risk_flag
0,2026-08-10,SPY,0.128,0.312,normal
1,2026-08-11,SPY,0.124,0.298,normal
2,2026-08-12,SPY,0.121,0.284,normal
3,2026-08-13,SPY,0.117,0.271,normal
4,2026-08-14,SPY,0.114,0.259,normal
5,2026-08-17,SPY,0.113,0.252,normal


## 2) Save CSV to data/raw/ and Parquet to data/processed/
- Use timestamped filenames.
- Handle missing Parquet engine gracefully.

In [5]:
def ts() -> str:
    """Return a timestamp safe for the required filenames."""

    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")


snapshot_timestamp = ts()
csv_path = RAW / f"sample_{snapshot_timestamp}.csv"
pq_path = PROC / f"sample_{snapshot_timestamp}.parquet"

df.to_csv(csv_path, index=False)
try:
    df.to_parquet(pq_path, index=False)
except ImportError as exc:
    raise RuntimeError(
        "Parquet engine not available. Install pyarrow or fastparquet."
    ) from exc

print("CSV saved to:", csv_path)
print("Parquet saved to:", pq_path)

CSV saved to: data/raw/sample_20260818-193918.csv
Parquet saved to: data/processed/sample_20260818-193918.parquet


## 3) Reload and validate
The CSV reload below intentionally uses the default reader without `parse_dates`. CSV stores text rather than pandas dtype metadata, so `date` returns as `object` unless it is parsed and categorical columns also return as `object`. Parquet preserves the datetime, float, and categorical dtypes. This contrast is the main storage decision demonstrated in this stage.

In [6]:
def validate_loaded(
    original: pd.DataFrame, reloaded: pd.DataFrame
) -> dict[str, bool]:
    """Compare shape and critical dtypes after a storage round-trip."""

    critical_columns = {
        "date",
        "next_five_session_vol_forecast",
        "risk_flag",
    }
    columns_present = critical_columns.issubset(reloaded.columns)
    dtype_matches = {
        column: columns_present and original[column].dtype == reloaded[column].dtype
        for column in critical_columns
    }
    return {
        "shape_matches": original.shape == reloaded.shape,
        "column_order_matches": list(original.columns) == list(reloaded.columns),
        "critical_columns_present": columns_present,
        "date_dtype_matches": dtype_matches["date"],
        "forecast_float_dtype_matches": dtype_matches[
            "next_five_session_vol_forecast"
        ],
        "risk_flag_category_dtype_matches": dtype_matches["risk_flag"],
        "all_critical_dtypes_match": all(dtype_matches.values()),
    }


df_csv = pd.read_csv(csv_path)
csv_checks = validate_loaded(df, df_csv)
print("CSV validation:", csv_checks)
assert csv_checks["shape_matches"]

CSV validation: {'shape_matches': True, 'column_order_matches': True, 'critical_columns_present': True, 'date_dtype_matches': False, 'forecast_float_dtype_matches': True, 'risk_flag_category_dtype_matches': False, 'all_critical_dtypes_match': False}


In [7]:
try:
    df_pq = pd.read_parquet(pq_path)
except ImportError as exc:
    raise RuntimeError(
        "Parquet engine not available. Install pyarrow or fastparquet."
    ) from exc

parquet_checks = validate_loaded(df, df_pq)
print("Parquet validation:", parquet_checks)
assert parquet_checks["shape_matches"]
assert parquet_checks["all_critical_dtypes_match"]

Parquet validation: {'shape_matches': True, 'column_order_matches': True, 'critical_columns_present': True, 'date_dtype_matches': True, 'forecast_float_dtype_matches': True, 'risk_flag_category_dtype_matches': True, 'all_critical_dtypes_match': True}


## 4) Storage utilities
The helpers route by suffix, create missing parent directories, and raise a targeted message only when a Parquet engine is unavailable. CSV reloads remain unparsed so the format's dtype limitation stays visible.

In [8]:
PARQUET_SUFFIXES = {".parquet", ".pq", ".parq"}
PARQUET_ERROR = "Parquet engine not available. Install pyarrow or fastparquet."


def detect_format(path: str | Path) -> str:
    """Return the storage format implied by a file suffix."""

    suffix = Path(path).suffix.lower()
    if suffix == ".csv":
        return "csv"
    if suffix in PARQUET_SUFFIXES:
        return "parquet"
    raise ValueError(f"Unsupported dataframe format: {suffix or '<none>'}")


def parquet_engine_available() -> bool:
    """Return whether pandas can use a supported Parquet engine."""

    return any(
        importlib.util.find_spec(engine) is not None
        for engine in ("pyarrow", "fastparquet")
    )


def require_parquet_engine() -> None:
    """Raise the assignment's clear error when no engine is installed."""

    if not parquet_engine_available():
        raise RuntimeError(PARQUET_ERROR)


def write_df(frame: pd.DataFrame, path: str | Path) -> Path:
    """Write a DataFrame as CSV or Parquet based on its suffix."""

    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    file_format = detect_format(destination)
    if file_format == "csv":
        frame.to_csv(destination, index=False)
    else:
        require_parquet_engine()
        try:
            frame.to_parquet(destination, index=False)
        except ImportError as exc:
            raise RuntimeError(PARQUET_ERROR) from exc
    return destination


def read_df(path: str | Path) -> pd.DataFrame:
    """Read a DataFrame as CSV or Parquet based on its suffix."""

    source = Path(path)
    file_format = detect_format(source)
    if file_format == "csv":
        return pd.read_csv(source)
    require_parquet_engine()
    try:
        return pd.read_parquet(source)
    except ImportError as exc:
        raise RuntimeError(PARQUET_ERROR) from exc


utility_timestamp = ts()
utility_csv_path = RAW / f"utility_sample_{utility_timestamp}.csv"
utility_pq_path = PROC / f"utility_sample_{utility_timestamp}.parquet"

write_df(df, utility_csv_path)
write_df(df, utility_pq_path)
utility_csv = read_df(utility_csv_path)
utility_pq = read_df(utility_pq_path)

print("Utility CSV validation:", validate_loaded(df, utility_csv))
print("Utility Parquet validation:", validate_loaded(df, utility_pq))
try:
    detect_format("data/processed/sample.json")
except ValueError as exc:
    print("Unsupported suffix check:", exc)

Utility CSV validation: {'shape_matches': True, 'column_order_matches': True, 'critical_columns_present': True, 'date_dtype_matches': False, 'forecast_float_dtype_matches': True, 'risk_flag_category_dtype_matches': False, 'all_critical_dtypes_match': False}
Utility Parquet validation: {'shape_matches': True, 'column_order_matches': True, 'critical_columns_present': True, 'date_dtype_matches': True, 'forecast_float_dtype_matches': True, 'risk_flag_category_dtype_matches': True, 'all_critical_dtypes_match': True}
Unsupported suffix check: Unsupported dataframe format: .json


## 5) Documentation
The README's **Data Storage** section documents the raw and processed folders, the reasons for using CSV and Parquet, and the environment-driven read/write flow. The checks above confirm that both shapes survive the round-trip, while only Parquet preserves every critical dtype without parsing instructions.

The course structure does not assign a `src/` folder to homework05, so the storage utilities stay in this notebook for this stage. In the cumulative Weekly ETF Risk Monitor, they graduate into `project/src/storage.py` for reuse by the project pipeline.